#  Price Recommendation Engine

##  Goal
Build a smart pricing system that recommends the **optimal product price** based on:

- User behavior
- Product statistics
- Purchase probability
- Predicted price

##  Approach
We combine:
1. **Classification Model** → Predict purchase likelihood
2. **Regression Model** → Predict price
3. **Business Logic** → Adjust price dynamically

Final Output:
 Recommended Price for each product-user interaction

In [ ]:
import pandas as pd
import numpy as np
import joblib

##  Load Processed Data
We use the final feature-engineered dataset.

In [ ]:
df = pd.read_csv(r"C:\Users\hp\Desktop\PROJECTS\Dynamic_Pricing_Engine\data\final\final_featured_data.csv")
df.head()

##  Load Models

- Classification → Purchase prediction
- Regression → Price prediction

In [ ]:
clf_model = joblib.load(r"C:\Users\hp\Desktop\PROJECTS\Dynamic_Pricing_Engine\models\classification_model.pkl")
reg_model = joblib.load(r"C:\Users\hp\Desktop\PROJECTS\Dynamic_Pricing_Engine\models\regression_model.pkl")

##  Feature Selection

Separate features for:
- Classification
- Regression

In [ ]:
X = df.drop(columns=['is_purchase', 'log_price'])

X_class = X.copy()
X_reg = X.copy()

##  Step 1: Predict Purchase Probability

This tells us:
 How likely a user is to buy

In [ ]:
#recreate log_price 
df['log_price'] = np.log1p(df['price'])

#using SAME logic as training
X_class = df.drop(columns=['is_purchase', 'price'])

In [ ]:
purchase_prob = clf_model.predict_proba(X_class)[:, 1]

df['purchase_prob'] = purchase_prob

##  Step 2: Predict Optimal Price

We predict log price and convert back to actual price.

In [ ]:
X_reg = df.drop(columns=['log_price', 'price', 'purchase_prob'])


In [ ]:
# Predict log price
log_price_pred = reg_model.predict(X_reg)

# Convert to actual price
price_pred = np.expm1(log_price_pred)

# Store result
df['predicted_price'] = price_pred

##  Step 3: Smart Dynamic Pricing

We combine:
- Purchase probability (demand)
- Predicted price (value)

### Strategy:
- High demand → increase price
- Low demand → discount
- Medium demand → slight optimization

In [ ]:
def smart_pricing(row):
    base_price = row['predicted_price']
    prob = row['purchase_prob']
    
    # very High demand 
    if prob > 0.8:
        return base_price * 1.15 #15percent increase

    #High Demand
    elif prob > 0.6:
        return base_price * 1.08  #8 percent increase 

    #Medium Demand
    elif prob > 0.4:
        return base_price #no change

    #Low Demand
    elif prob > 0.2:
        return base_price * 0.92  #8percent discount

    #Very Low Demand
    else:
        return base_price * 0.85   #15 percent discount

#output stored
df['recommended_price'] = df.apply(smart_pricing, axis=1)